
# Notebook 19 — Residual Phase Trajectories

This version is fully self-contained for Colab.

It will:

1. try to find repo-level `results/`,
2. load Notebook 17/18 residual feature files if present,
3. regenerate compatible residual geometry features if files are missing,
4. compute residual phase trajectories.

Core question:

```text
How do topology classes move through residual manifold space as graph size increases?
```

Core claim:

```text
Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.
```


In [ ]:

import json
import zipfile
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)

# ---------------------------------------------------
# robust folders for Colab and local repo use
# ---------------------------------------------------

def detect_repo_root():
    cwd = Path.cwd()

    # already in repo root
    if (cwd / "notebooks").exists() or (cwd / ".git").exists():
        return cwd

    # running from repo/notebooks
    if cwd.name == "notebooks":
        return cwd.parent

    # search /content for known repo outputs
    search_root = Path("/content") if Path("/content").exists() else cwd
    for name in [
        "residual_classification_feature_matrix.csv",
        "residual_geometry_features.csv",
        "residual_field_data.csv",
    ]:
        matches = list(search_root.rglob(name))
        if matches:
            return matches[0].parents[1]  # repo/results/file.csv -> repo

    # fallback to /content if Colab
    return Path("/content") if Path("/content").exists() else cwd

REPO_ROOT = detect_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

GRAPH_SIZES = [16, 32, 64, 128]
TOPOLOGIES = [
    "ring_lattice",
    "small_world",
    "erdos_renyi",
    "scale_free",
    "modular_clustered",
]

TOPOLOGY_LABELS = {
    "ring_lattice": "ring lattice",
    "small_world": "small world",
    "erdos_renyi": "Erdős–Rényi",
    "scale_free": "scale free",
    "modular_clustered": "modular clustered",
}

NOISE_GRID = np.linspace(0.0, 0.40, 81)
MIDPOINT_LEVEL = 0.50

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("results dir:", RESULTS_DIR)
print("Ready.")


## Shared helpers

In [ ]:

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def shared_profile(z):
    return 1 - logistic_z(z)

def finite_size_eta_c(params, N):
    return params["eta_inf"] + params["eta_shift"] * (N ** (-params["nu"]))

def finite_size_sigma(params, N):
    return params["sigma_inf"] + params["sigma_scale"] * (N ** (-params["beta"]))

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135,
        "eta_shift": 0.060,
        "sigma_inf": 0.030,
        "sigma_scale": 0.080,
        "nu": 0.45,
        "beta": 0.40,
        "fragment_strength": 0.08,
        "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160,
        "eta_shift": 0.075,
        "sigma_inf": 0.038,
        "sigma_scale": 0.095,
        "nu": 0.48,
        "beta": 0.37,
        "fragment_strength": 0.06,
        "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120,
        "eta_shift": 0.055,
        "sigma_inf": 0.028,
        "sigma_scale": 0.075,
        "nu": 0.42,
        "beta": 0.45,
        "fragment_strength": 0.10,
        "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105,
        "eta_shift": 0.052,
        "sigma_inf": 0.025,
        "sigma_scale": 0.070,
        "nu": 0.44,
        "beta": 0.50,
        "fragment_strength": 0.14,
        "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092,
        "eta_shift": 0.048,
        "sigma_inf": 0.023,
        "sigma_scale": 0.065,
        "nu": 0.40,
        "beta": 0.52,
        "fragment_strength": 0.18,
        "modifier": 0.90,
    },
}

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]
    eta_c = finite_size_eta_c(p, N)
    sigma = finite_size_sigma(p, N)

    z = (noise_grid - eta_c) / sigma
    base = shared_profile(z)

    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight
    fragment = (
        p["fragment_strength"]
        * outside_weight
        * logistic_z((noise_grid - eta_c) / (2.0 * sigma))
    )

    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    return np.clip(p["modifier"] * base - fragment + noise_term, 0, 1)

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    deriv = np.gradient(cgcs, noise)
    max_abs_slope = float(np.max(np.abs(deriv)))
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return eta_mid, sigma_est

def smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3):
    ordered = sub.sort_values("z")
    tmp = (
        pd.DataFrame({
            "z": ordered["z"].to_numpy(dtype=float),
            "r": ordered["residual"].to_numpy(dtype=float),
        })
        .groupby("z", as_index=False)
        .mean()
    )

    z = tmp["z"].to_numpy()
    r = tmp["r"].to_numpy()

    if len(z) < 5:
        return np.full_like(z_grid, np.nan), np.full_like(z_grid, np.nan)

    r_grid = np.interp(z_grid, z, r, left=np.nan, right=np.nan)
    valid = np.isfinite(r_grid)

    if valid.sum() < window:
        return r_grid, r_grid

    r_valid = r_grid[valid]
    w = min(window, len(r_valid) if len(r_valid) % 2 == 1 else len(r_valid) - 1)
    w = max(w, polyorder + 2)
    if w % 2 == 0:
        w -= 1

    if w <= polyorder:
        return r_grid, r_grid

    smooth_valid = savgol_filter(r_valid, window_length=w, polyorder=polyorder, mode="interp")
    smooth_grid = r_grid.copy()
    smooth_grid[valid] = smooth_valid

    return r_grid, smooth_grid

def normalized_entropy_from_energy(z, energy, bins=24):
    hist, _ = np.histogram(z, bins=bins, range=(-6, 6), weights=energy)
    total = hist.sum()

    if total <= 0:
        return 0.0

    p = hist / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)) / np.log(bins))


## Load or regenerate residual geometry features

In [ ]:

def regenerate_residual_field_data():
    rows = []
    repeats = 24

    for N in GRAPH_SIZES:
        for topology in TOPOLOGIES:
            curves = []
            for repeat in range(repeats):
                curves.append(simulate_cgcs_curve(topology, N, NOISE_GRID, repeat))

            mean_curve = np.array(curves).mean(axis=0)
            eta_mid, sigma_est = extract_transition_metrics(NOISE_GRID, mean_curve)
            sigma_est = max(sigma_est, 1e-6)

            for eta, cgcs in zip(NOISE_GRID, mean_curve):
                z = (eta - eta_mid) / sigma_est
                sp = shared_profile(z)
                residual = float(cgcs - sp)

                rows.append({
                    "topology": topology,
                    "label": TOPOLOGY_LABELS[topology],
                    "n_modules": int(N),
                    "link_noise": float(eta),
                    "cgcs": float(cgcs),
                    "z": float(z),
                    "shared_profile_recomputed": float(sp),
                    "residual": residual,
                    "abs_residual": float(abs(residual)),
                    "residual_energy": float(residual ** 2),
                })

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "residual_field_data.csv", index=False)
    return df

def compute_residual_geometry_features(collapse_df):
    rows = []

    z_grid = np.linspace(-6, 6, 241)
    z_fft = np.linspace(-6, 6, 256)

    for (topology, N), sub in collapse_df.groupby(["topology", "n_modules"]):
        energy = sub["residual_energy"].to_numpy()
        abs_res = sub["abs_residual"].to_numpy()
        total_energy = float(np.sum(energy))

        if total_energy <= 0:
            residual_localization = 0.0
        else:
            k = max(1, int(np.ceil(0.10 * len(energy))))
            residual_localization = float(np.sort(energy)[-k:].sum() / total_energy)

        left = sub[sub["z"] < 0]
        right = sub[sub["z"] >= 0]
        left_energy = float(left["residual_energy"].sum())
        right_energy = float(right["residual_energy"].sum())
        lr_total = left_energy + right_energy
        residual_asymmetry = float((right_energy - left_energy) / lr_total) if lr_total > 0 else 0.0

        residual_entropy = normalized_entropy_from_energy(
            sub["z"].to_numpy(),
            sub["residual_energy"].to_numpy(),
            bins=24,
        )

        _, smooth_grid = smooth_residual_on_grid(sub, z_grid, window=17, polyorder=3)
        mask = np.isfinite(smooth_grid)

        if mask.sum() >= 7:
            z_valid = z_grid[mask]
            r_smooth = smooth_grid[mask]
            first = np.gradient(r_smooth, z_valid)
            second = np.gradient(first, z_valid)
            residual_bend_energy = float(np.trapz(second ** 2, z_valid))
            mean_abs_bend = float(np.mean(np.abs(second)))
        else:
            residual_bend_energy = np.nan
            mean_abs_bend = np.nan

        _, smooth_fft = smooth_residual_on_grid(sub, z_fft, window=17, polyorder=3)
        valid = np.isfinite(smooth_fft)

        if valid.sum() >= 8:
            fill = np.nanmean(smooth_fft)
            r_grid = np.where(valid, smooth_fft, fill)
            r_centered = r_grid - np.mean(r_grid)
            power = np.abs(np.fft.rfft(r_centered)) ** 2
            power[0] = 0
            total_power = float(power.sum())

            if total_power > 0:
                cutoff = max(2, int(0.20 * len(power)))
                low_energy = float(power[1:cutoff].sum() / total_power)
                high_energy = float(power[cutoff:].sum() / total_power)
                residual_spectral_ratio = float(high_energy / max(low_energy, 1e-9))
            else:
                residual_spectral_ratio = 0.0
        else:
            residual_spectral_ratio = np.nan

        rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "n_modules": int(N),
            "mean_abs_residual": float(np.mean(abs_res)),
            "max_abs_residual": float(np.max(abs_res)),
            "total_residual_energy": total_energy,
            "residual_localization": residual_localization,
            "residual_asymmetry": residual_asymmetry,
            "residual_entropy": residual_entropy,
            "residual_bend_energy": residual_bend_energy,
            "mean_abs_bend": mean_abs_bend,
            "residual_spectral_ratio": residual_spectral_ratio,
        })

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_DIR / "residual_geometry_features.csv", index=False)
    return df

feature_path = RESULTS_DIR / "residual_classification_feature_matrix.csv"
geometry_path = RESULTS_DIR / "residual_geometry_features.csv"
field_path = RESULTS_DIR / "residual_field_data.csv"
loso_path = RESULTS_DIR / "residual_leave_one_size_out_predictions.csv"

if feature_path.exists():
    feature_df = pd.read_csv(feature_path)
    data_source = "loaded Notebook 18 feature matrix"
elif geometry_path.exists():
    feature_df = pd.read_csv(geometry_path)
    data_source = "loaded Notebook 17 geometry features"
else:
    residual_field_df = regenerate_residual_field_data()
    feature_df = compute_residual_geometry_features(residual_field_df)
    data_source = "regenerated residual geometry features internally"

feature_df = feature_df.replace([np.inf, -np.inf], np.nan).dropna()
feature_df["label"] = feature_df["topology"].map(TOPOLOGY_LABELS)
feature_df = feature_df.sort_values(["topology", "n_modules"]).reset_index(drop=True)

print("data source:", data_source)
print("feature_df shape:", feature_df.shape)
feature_df.head()


## Recompute a shared PCA manifold

In [ ]:

CANDIDATE_FEATURES = [
    "mean_abs_residual",
    "max_abs_residual",
    "total_residual_energy",
    "residual_localization",
    "residual_asymmetry",
    "residual_entropy",
    "residual_bend_energy",
    "mean_abs_bend",
    "residual_spectral_ratio",
]

feature_cols = [c for c in CANDIDATE_FEATURES if c in feature_df.columns]

X = feature_df[feature_cols].to_numpy(dtype=float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
Z = pca.fit_transform(X_scaled)

traj_df = feature_df[["topology", "label", "n_modules"]].copy()
traj_df["pc1"] = Z[:, 0]
traj_df["pc2"] = Z[:, 1]

traj_df.to_csv(RESULTS_DIR / "residual_phase_trajectory_embedding.csv", index=False)

print("features:", feature_cols)
print("PCA explained variance:", pca.explained_variance_ratio_)
traj_df.head()


## Figure 1 — Topology trajectory paths

In [ ]:

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    if sub.empty:
        continue

    plt.plot(
        sub["pc1"],
        sub["pc2"],
        marker="o",
        linewidth=2,
        markersize=8,
        label=TOPOLOGY_LABELS[topology],
    )

    for i in range(len(sub) - 1):
        x0, y0 = sub.iloc[i][["pc1", "pc2"]]
        x1, y1 = sub.iloc[i + 1][["pc1", "pc2"]]
        plt.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.7),
        )

    for _, row in sub.iterrows():
        plt.annotate(
            f"N={int(row['n_modules'])}",
            xy=(row["pc1"], row["pc2"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            alpha=0.8,
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title("Residual phase trajectories across graph size")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "topology_trajectory_pca.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Trajectory metrics

In [ ]:

def path_length(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return 0.0
    return float(np.sum(np.linalg.norm(np.diff(points, axis=0), axis=1)))

def path_curvature(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 3:
        return 0.0
    second = points[2:] - 2 * points[1:-1] + points[:-2]
    return float(np.sum(np.linalg.norm(second, axis=1)))

def endpoint_vector(points):
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return np.zeros(2)
    return points[-1] - points[0]

trajectory_rows = []

for topology in TOPOLOGIES:
    sub = traj_df[traj_df["topology"] == topology].sort_values("n_modules")
    points = sub[["pc1", "pc2"]].to_numpy(dtype=float)

    L = path_length(points)
    K = path_curvature(points)
    v = endpoint_vector(points)
    straight = float(np.linalg.norm(v))
    tortuosity = float(L / max(straight, 1e-9))

    trajectory_rows.append({
        "topology": topology,
        "label": TOPOLOGY_LABELS[topology],
        "n_points": int(len(points)),
        "trajectory_length": L,
        "endpoint_distance": straight,
        "trajectory_curvature": K,
        "trajectory_tortuosity": tortuosity,
        "endpoint_dx": float(v[0]),
        "endpoint_dy": float(v[1]),
    })

trajectory_metrics_df = pd.DataFrame(trajectory_rows)
trajectory_metrics_df.to_csv(RESULTS_DIR / "residual_phase_trajectory_metrics.csv", index=False)

trajectory_metrics_df


In [ ]:

plot_df = trajectory_metrics_df.sort_values("trajectory_length", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["label"], plot_df["trajectory_length"])
plt.xlabel("PCA trajectory length")
plt.title("Residual manifold trajectory length by topology")
plt.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "trajectory_length_by_topology.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


In [ ]:

plot_df = trajectory_metrics_df.sort_values("trajectory_curvature", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_df["label"], plot_df["trajectory_curvature"])
plt.xlabel("trajectory curvature")
plt.title("Residual trajectory curvature by topology")
plt.grid(alpha=0.3, axis="x")

fig_path = FIG_DIR / "trajectory_curvature.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Direction alignment matrix

In [ ]:

vectors = {}
for topology in TOPOLOGIES:
    row = trajectory_metrics_df[trajectory_metrics_df["topology"] == topology].iloc[0]
    vectors[topology] = np.array([row["endpoint_dx"], row["endpoint_dy"]], dtype=float)

alignment = np.zeros((len(TOPOLOGIES), len(TOPOLOGIES)))

for i, t1 in enumerate(TOPOLOGIES):
    for j, t2 in enumerate(TOPOLOGIES):
        u = vectors[t1]
        v = vectors[t2]
        denom = np.linalg.norm(u) * np.linalg.norm(v)
        alignment[i, j] = float(np.dot(u, v) / denom) if denom > 0 else np.nan

alignment_df = pd.DataFrame(
    alignment,
    index=[TOPOLOGY_LABELS[t] for t in TOPOLOGIES],
    columns=[TOPOLOGY_LABELS[t] for t in TOPOLOGIES],
)

alignment_out = alignment_df.copy()
alignment_out.insert(0, "topology", alignment_out.index)
alignment_out.to_csv(RESULTS_DIR / "residual_direction_alignment.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(alignment, vmin=-1, vmax=1)

ax.set_xticks(range(len(TOPOLOGIES)))
ax.set_yticks(range(len(TOPOLOGIES)))
ax.set_xticklabels([TOPOLOGY_LABELS[t] for t in TOPOLOGIES], rotation=45, ha="right")
ax.set_yticklabels([TOPOLOGY_LABELS[t] for t in TOPOLOGIES])

for i in range(len(TOPOLOGIES)):
    for j in range(len(TOPOLOGIES)):
        ax.text(j, i, f"{alignment[i, j]:.2f}", ha="center", va="center")

ax.set_title("Residual trajectory direction alignment")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine alignment")
plt.tight_layout()

fig_path = FIG_DIR / "direction_alignment_matrix.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Radial distance vs graph size

In [ ]:

center = traj_df[["pc1", "pc2"]].mean().to_numpy(dtype=float)

radial_df = traj_df.copy()
radial_df["radial_distance"] = np.linalg.norm(
    radial_df[["pc1", "pc2"]].to_numpy(dtype=float) - center,
    axis=1,
)

radial_df.to_csv(RESULTS_DIR / "residual_radial_growth.csv", index=False)

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = radial_df[radial_df["topology"] == topology].sort_values("n_modules")
    plt.plot(
        sub["n_modules"],
        sub["radial_distance"],
        marker="o",
        linewidth=2,
        label=TOPOLOGY_LABELS[topology],
    )

plt.xlabel("graph size N")
plt.ylabel("distance from global residual centroid")
plt.title("Residual radial distance vs graph size")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "radial_distance_vs_size.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Phase drift vs classification stability

In [ ]:

loso_path = RESULTS_DIR / "residual_leave_one_size_out_predictions.csv"

if loso_path.exists():
    loso_predictions = pd.read_csv(loso_path)

    stability_rows = []
    for topology in TOPOLOGIES:
        sub = loso_predictions[loso_predictions["topology"] == topology]
        acc = float(sub["loso_correct"].mean()) if "loso_correct" in sub.columns and len(sub) > 0 else np.nan

        traj = trajectory_metrics_df[trajectory_metrics_df["topology"] == topology].iloc[0]

        stability_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "trajectory_length": float(traj["trajectory_length"]),
            "trajectory_curvature": float(traj["trajectory_curvature"]),
            "leave_one_size_out_accuracy": acc,
        })

    stability_note = "loaded Notebook 18 leave-one-size-out predictions"
else:
    stability_rows = []
    for topology in TOPOLOGIES:
        traj = trajectory_metrics_df[trajectory_metrics_df["topology"] == topology].iloc[0]
        stability_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "trajectory_length": float(traj["trajectory_length"]),
            "trajectory_curvature": float(traj["trajectory_curvature"]),
            "leave_one_size_out_accuracy": np.nan,
        })
    stability_note = "Notebook 18 leave-one-size-out predictions unavailable"

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(RESULTS_DIR / "residual_phase_drift_vs_accuracy.csv", index=False)

plt.figure(figsize=(8, 6))

valid = stability_df.dropna(subset=["leave_one_size_out_accuracy"])
if len(valid) > 0:
    plt.scatter(valid["trajectory_length"], valid["leave_one_size_out_accuracy"], s=140, alpha=0.75)

    for _, row in valid.iterrows():
        plt.annotate(
            row["label"],
            xy=(row["trajectory_length"], row["leave_one_size_out_accuracy"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
        )

    plt.ylim(-0.05, 1.05)
else:
    plt.text(
        0.5, 0.5,
        "Notebook 18 leave-one-size-out predictions unavailable",
        ha="center",
        va="center",
        transform=plt.gca().transAxes,
    )

plt.xlabel("residual trajectory length")
plt.ylabel("leave-one-size-out accuracy")
plt.title("Phase drift vs topology classification stability")
plt.grid(alpha=0.3)

fig_path = FIG_DIR / "phase_drift_vs_accuracy.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()

print(stability_note)
stability_df


## Centroid flow after size-centering

In [ ]:

centroid_rows = []

for N, subN in traj_df.groupby("n_modules"):
    center_N = subN[["pc1", "pc2"]].mean().to_numpy(dtype=float)

    for topology in TOPOLOGIES:
        sub = subN[subN["topology"] == topology]
        if sub.empty:
            continue

        p = sub[["pc1", "pc2"]].iloc[0].to_numpy(dtype=float)
        centroid_rows.append({
            "topology": topology,
            "label": TOPOLOGY_LABELS[topology],
            "n_modules": int(N),
            "pc1": float(p[0]),
            "pc2": float(p[1]),
            "centered_pc1": float(p[0] - center_N[0]),
            "centered_pc2": float(p[1] - center_N[1]),
            "distance_from_size_centroid": float(np.linalg.norm(p - center_N)),
        })

centroid_flow_df = pd.DataFrame(centroid_rows)
centroid_flow_df.to_csv(RESULTS_DIR / "residual_centroid_flow.csv", index=False)

plt.figure(figsize=(10, 8))

for topology in TOPOLOGIES:
    sub = centroid_flow_df[centroid_flow_df["topology"] == topology].sort_values("n_modules")
    plt.plot(
        sub["centered_pc1"],
        sub["centered_pc2"],
        marker="o",
        linewidth=2,
        label=TOPOLOGY_LABELS[topology],
    )

    for i in range(len(sub) - 1):
        x0, y0 = sub.iloc[i][["centered_pc1", "centered_pc2"]]
        x1, y1 = sub.iloc[i + 1][["centered_pc1", "centered_pc2"]]
        plt.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.7),
        )

plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.5)
plt.xlabel("PC1 relative to size centroid")
plt.ylabel("PC2 relative to size centroid")
plt.title("Residual centroid flow after size-centering")
plt.grid(alpha=0.3)
plt.legend(fontsize=9)

fig_path = FIG_DIR / "topology_centroid_flow.png"
plt.savefig(fig_path, dpi=240, bbox_inches="tight")
plt.show()


## Summary export

In [ ]:

max_length_row = trajectory_metrics_df.sort_values("trajectory_length", ascending=False).iloc[0]
min_length_row = trajectory_metrics_df.sort_values("trajectory_length", ascending=True).iloc[0]
max_curv_row = trajectory_metrics_df.sort_values("trajectory_curvature", ascending=False).iloc[0]

summary = {
    "notebook": "19_residual_phase_trajectories_self_contained.ipynb",
    "core_question": "How do topology classes move through residual manifold space as graph size increases?",
    "core_claim": "Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.",
    "data_source": data_source,
    "features": feature_cols,
    "pca_explained_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "largest_trajectory_length": {
        "topology": str(max_length_row["topology"]),
        "label": str(max_length_row["label"]),
        "value": float(max_length_row["trajectory_length"]),
    },
    "smallest_trajectory_length": {
        "topology": str(min_length_row["topology"]),
        "label": str(min_length_row["label"]),
        "value": float(min_length_row["trajectory_length"]),
    },
    "largest_trajectory_curvature": {
        "topology": str(max_curv_row["topology"]),
        "label": str(max_curv_row["label"]),
        "value": float(max_curv_row["trajectory_curvature"]),
    },
    "figures": [
        "topology_trajectory_pca.png",
        "trajectory_length_by_topology.png",
        "trajectory_curvature.png",
        "direction_alignment_matrix.png",
        "radial_distance_vs_size.png",
        "phase_drift_vs_accuracy.png",
        "topology_centroid_flow.png",
    ],
    "results": [
        "residual_phase_trajectory_embedding.csv",
        "residual_phase_trajectory_metrics.csv",
        "residual_direction_alignment.csv",
        "residual_radial_growth.csv",
        "residual_phase_drift_vs_accuracy.csv",
        "residual_centroid_flow.csv",
        "residual_phase_trajectory_summary.json",
    ],
}

summary_path = RESULTS_DIR / "residual_phase_trajectory_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 19 — Residual Phase Trajectories",
    "",
    "**Core question:** How do topology classes move through residual manifold space as graph size increases?",
    "",
    "**Core claim:** Residual topology identity is not static; each topology traces a scale-dependent path through residual manifold space.",
    "",
    "Recommended paper figures:",
    "",
    "- `figures/topology_trajectory_pca.png`",
    "- `figures/trajectory_length_by_topology.png`",
    "- `figures/direction_alignment_matrix.png`",
    "- `figures/topology_centroid_flow.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_19_residual_phase_trajectories.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")


## Optional export zip

In [ ]:

zip_path = REPO_ROOT / "notebook_19_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
